In [13]:
import ray
import requests
import os
import hashlib
import io
import pandas as pd

In [14]:
def callMovebankAPI(params, user, pw):
    response = requests.get(
        'https://www.movebank.org/movebank/service/direct-read', 
        params=params, 
        auth=(user, pw)
    )
    
    if response.status_code == 200:
        if 'License Terms:' in str(response.content):
            hash_val = hashlib.md5(response.content).hexdigest()
            params = params + (('license-md5', hash_val),)
            
            response = requests.get(
                'https://www.movebank.org/movebank/service/direct-read', 
                params=params,
                cookies=response.cookies, 
                auth=(user, pw)
            )
            
            if response.status_code == 403:
                return ''
                
        return response.content.decode('utf-8')
        
    return ''

In [15]:
@ray.remote(num_cpus=1)
def download_study_ray(study_id, study_name, user, pw, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    file_path = os.path.join(output_dir, f"study_{study_id}.parquet")
    
    if os.path.exists(file_path):
        return f"SKIP: Studie {study_id} existiert bereits."

    params = (
        ('entity_type', 'event'),
        ('study_id', str(study_id)),
        ('sensor_type_id', '653'), 
        ('attributes', 'timestamp,location_long,location_lat,individual_local_identifier,ground_speed,heading,visible')
    )

    csv_data = callMovebankAPI(params, user, pw)

    if csv_data and "timestamp" in csv_data[:100]:
        try:
            df = pd.read_csv(io.StringIO(csv_data))
            if not df.empty:
                df.to_parquet(file_path, index=False)
                return f"SUCCESS: {len(df)} Punkte fuer '{study_name[:30]}...' gespeichert."
            else:
                return f"EMPTY: Studie {study_id} enthaelt keine GPS-Daten."
        except Exception as e:
            return f"ERROR: Fehler beim Parsen von Studie {study_id}: {e}"
    else:
        return f"PRIVATE: Kein Zugriff auf Studie {study_id}."

In [18]:
def get_taxon_inventory(taxa_list, user, pw):
    print("Erstelle Inventar: Lade globale Studien-Zusammenfassungen...")
    
    params_study = (
        ('entity_type', 'study'),
        ('i_can_see_data', 'true')
    )
    
    csv_data = callMovebankAPI(params_study, user, pw)
    if not csv_data:
        print("Konnte keine Studien abrufen.")
        return pd.DataFrame()

    try:
        df_studies = pd.read_csv(io.StringIO(csv_data))
    except Exception as e:
        print(f"Fehler beim Parsen der Studien: {e}")
        return pd.DataFrame()

    # Wir nutzen die Liste, um in allen Textfeldern der Studie zu suchen
    pattern = '|'.join(taxa_list)
    
    # Filtert alle Studien, bei denen die Zielwoerter irgendwo in den Metadaten auftauchen
    inventory_df = df_studies[df_studies.apply(lambda row: row.astype(str).str.contains(pattern, case=False).any(), axis=1)].copy()
    
    print(f"Erfolg: {len(inventory_df)} relevante Studien fuer den Download markiert.")
    
    return inventory_df

In [19]:

# Trage hier noch einmal deine echten Daten ein, damit der Head-Node sie an die Worker verteilen kann
MBUS = "segmumik"
MBPW = "xuzmux0cenmyfomHyb"

CLUSTER_STORAGE_DIR = "/mnt/shared_data/finflow/mvbk_raw"
INVENTORY_FILE = os.path.join(CLUSTER_STORAGE_DIR, "movebank_finflow_inventory.csv")

target_taxa = ["Cetacea", "Phocidae", "Cheloniidae", "Whale", "Seal", "Turtle"]

print("Initialisiere Ray-Cluster...")
if not ray.is_initialized():
    # Passe dies an, falls du das Skript lokal testest (ray.init())
    ray.init(address='auto') 

# 1. Pruefen, ob das Inventar existiert, sonst erstellen
if not os.path.exists(INVENTORY_FILE):
    print(f"Inventar-Datei '{INVENTORY_FILE}' nicht gefunden. Starte Neuerstellung...")
    os.makedirs(CLUSTER_STORAGE_DIR, exist_ok=True)
    
    inventory_df = get_taxon_inventory(target_taxa, MBUS, MBPW)
    
    if not inventory_df.empty:
        inventory_df = inventory_df.drop_duplicates(subset=['id'])
        inventory_df.to_csv(INVENTORY_FILE, index=False)
        print(f"Inventar erfolgreich erstellt und unter {INVENTORY_FILE} gespeichert.")
    else:
        print("FEHLER: Konnte kein Inventar erstellen. Bitte Log-Ausgaben pruefen.")
        exit(1) # Hier ist ein Exit legitim, da ohne Inventar nichts heruntergeladen werden kann
else:
    print(f"Lade bestehendes Inventar von {INVENTORY_FILE}...")
    inventory_df = pd.read_csv(INVENTORY_FILE)
    
# 2. Verteilen der Aufgaben an die Ray-Worker
print(f"Starte verteilten Download fuer {len(inventory_df)} Studien...")

task_references = []
for index, row in inventory_df.iterrows():
    task_ref = download_study_ray.remote(
        study_id=row['id'], 
        study_name=row['name'], 
        user=MBUS, 
        pw=MBPW, 
        output_dir=CLUSTER_STORAGE_DIR
    )
    task_references.append(task_ref)
    
# Warten, bis alle Worker fertig sind
results = ray.get(task_references)

print("\nALLE DOWNLOADS ABGESCHLOSSEN:")

success_count = 0
private_count = 0
skip_count = 0
error_count = 0

for res in results:
    print(res)
    if "SUCCESS" in res:
        success_count += 1
    elif "PRIVATE" in res or "EMPTY" in res:
        private_count += 1
    elif "SKIP" in res:
        skip_count += 1
    elif "ERROR" in res:
        error_count += 1
        
print(f"\nZusammenfassung: {success_count} neu geladen, {skip_count} uebersprungen, {private_count} privat/leer, {error_count} Fehler.")

Initialisiere Ray-Cluster...
Inventar-Datei '/mnt/shared_data/finflow/mvbk_raw/movebank_finflow_inventory.csv' nicht gefunden. Starte Neuerstellung...
Erstelle Inventar: Lade globale Studien-Zusammenfassungen...
Erfolg: 126 relevante Studien fuer den Download markiert.
Inventar erfolgreich erstellt und unter /mnt/shared_data/finflow/mvbk_raw/movebank_finflow_inventory.csv gespeichert.
Starte verteilten Download fuer 126 Studien...

ALLE DOWNLOADS ABGESCHLOSSEN:
PRIVATE: Kein Zugriff auf Studie 7006760.
SKIP: Studie 259966228 existiert bereits.
PRIVATE: Kein Zugriff auf Studie 2768691184.
PRIVATE: Kein Zugriff auf Studie 1670575575.
PRIVATE: Kein Zugriff auf Studie 265607070.
PRIVATE: Kein Zugriff auf Studie 886013997.
PRIVATE: Kein Zugriff auf Studie 7355748409.
PRIVATE: Kein Zugriff auf Studie 5020584004.
PRIVATE: Kein Zugriff auf Studie 7689226744.
PRIVATE: Kein Zugriff auf Studie 136705437.
PRIVATE: Kein Zugriff auf Studie 2946547482.
EMPTY: Studie 72289508 enthaelt keine GPS-Daten.

(raylet) [2026-03-19 12:18:21,232 E 1839 1839] (raylet) node_manager.cc:3250: 6 Workers (tasks / actors) killed due to memory pressure (OOM), 0 Workers crashed due to other reasons at node (ID: 71a280497b5bd9d014baacef31ba8dfde802e39f435dcd959cb8bbe2, IP: 10.10.1.98) over the last time period. To see more information about the Workers killed on this node, use `ray logs raylet.out -ip 10.10.1.98`
(raylet) 
(raylet) Refer to the documentation on how to address the out of memory issue: https://docs.ray.io/en/latest/ray-core/scheduling/ray-oom-prevention.html. Consider provisioning more memory on this node or reducing task parallelism by requesting more CPUs per task. To adjust the kill threshold, set the environment variable `RAY_memory_usage_threshold` when starting Ray. To disable worker killing, set the environment variable `RAY_memory_monitor_refresh_ms` to zero.
